##**Identifying similar songs using real-world data**

**Dataset** = Spotify Tracks Dataset (https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)
- Open-source music database containing attributes for identifying type of music (inc. tempo, loudness, energy etc.)
- Designed primarily for data analysis, ML and recommendation system development



In [6]:
import pandas as pd

#Import Dataset from Kaggel
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")

# List the files in the directory to confirm the filename
print("Files:", os.listdir(path))

# Load the CSV file into pandas (usually named dataset.csv)
file_path = os.path.join(path, "dataset.csv")
data = pd.read_csv(file_path)

# View the first few ten rows
print(data.head())

# Check for missing values
print(data.isnull().sum())

Using Colab cache for faster access to the '-spotify-tracks-dataset' dataset.
Files: ['dataset.csv']
   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic      

####Classifying the genre of a target track based on the input features:  `tempo`, `energy` and `danceability`

- Select the `features` such as `tempo`, `energy` and `danceability`.
- Based on those `features`, use the KNN classifier to find similar `track_genre`

In [7]:
from sklearn.preprocessing import StandardScaler

# Select features for the recommendation
features = ['tempo', 'energy', 'danceability']

X = data[features]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Genre is the target
genres = data['track_genre']

**The following code uses** `NearestNeighbors`**, with the following outline:**

**Aim:** Find songs close to this one inside the same genre
1) Look up target song genre (eg. `jazz`)
2) Drop songs not of `jazz` genre
3) Calculate k-NN distances using audio features to find closest jazz songs to target song


In [8]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Fit a KNN model on the scaled features
knn = NearestNeighbors(n_neighbors=6, metric='euclidean')  # 6 because 1 will be the song itself
knn.fit(X_scaled)

def recommend_songs(song_index, n_recommendations=5):
    distances, indices = knn.kneighbors([X_scaled[song_index]], n_neighbors=n_recommendations+1)
    # Exclude the first index, as it is the song itself
    recommended_indices = indices[0][1:]
    recommended_songs = data.iloc[recommended_indices]
    #return recommended_songs[[ 'artists', 'loudness', 'popularity', 'liveness']]
    return recommended_songs[[ 'artists']]
# For example, recommend songs that are similar to the song at index 10
print(recommend_songs(10))

                   artists
10            Zack Tabudlo
710            Hanare Gumi
60267  Vipa Music;Kenyaman
55279                Papon
34030                Papon


> For the above, `model.kneighbours()` is used
>> the model is given in `knn = NearestNeighbours`, and `.kneighbours` gives the mathematical search to get the indices and calculated distances for the specific song

> Parameters for `.kneighbours`:
>> `X` ((array-like), shape, n_queries, n_features) \
>> `n_neighbours` (int, optional) \
>> `return_distance` (`True` = returns types containing two 2D arrays: `(distances, indices)`



**The following code uses** `NearestNeighbors`**, with the following outline:**

**Aim:** Find songs close to this one inside the same genre
1) Look up song of genre (eg. `jazz`)
2) Drop songs not of genre
3) Calculate k-NN distances using audio features to find closest songs of same genre to target song
4) Recommend songs within the genre with similar `'album_name', 'track_genre', 'tempo', 'energy'`


In [9]:
def recommend_songs_same_genre(song_index, n_recommendations=5):
    song_genre = genres.iloc[song_index] # find the genre of the target song
    genre_filter = data['track_genre'] == song_genre #get only the genre of the target song (creates Boolean mask identifying every song in dataset matching that genre)
    X_genre = X_scaled[genre_filter] #get songs of that genre
    knn_genre = NearestNeighbors(n_neighbors=n_recommendations+1)
    knn_genre.fit(X_genre)

    # Find the relative index of the song in the filtered data set
    filtered_indices = data[genre_filter].index.tolist()
    song_relative_index = filtered_indices.index(song_index)

    distances, indices = knn_genre.kneighbors([X_genre[song_relative_index]], n_neighbors=n_recommendations+1)
    recommended_indices = [filtered_indices[i] for i in indices[0][1:]]
    recommended_songs = data.loc[recommended_indices]
    return recommended_songs[['album_name','track_genre', 'tempo', 'energy']]

print(recommend_songs_same_genre(10))

                 album_name track_genre   tempo  energy
10                  Episode    acoustic  99.905   0.363
710                     深呼吸    acoustic  98.995   0.367
708           Guitar Covers    acoustic  95.981   0.346
199     Un tecito y a mimir    acoustic  97.498   0.399
972  Cover Sessions, Vol. 5    acoustic  95.988   0.336


**The following code uses** `kNeighborsClassifier`**, with the following outline:**

**Aim:** Predict what genre this song sounds like, then recommend that genre
1) Feed model 3 features (`popularity`, `liveness`, `tempo`)
2) Model looks at nearest neighbors to guess/classify the genre (eg. based on these 3 features it sounds like ‘indie’)
3) Find songs from that predicted genre, and recommend songs with similar `album_name`, `track_name`, `duration_ms`

In [10]:
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
#filter out by 3 features: popularity, liveness, tempo
nfeatures = ['popularity', 'liveness', 'tempo']
pre_nX = data[nfeatures]
nX = scaler.fit_transform(pre_nX)
y = data['track_genre']
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(nX, y)

def classify_and_recommend(song_index, req_features, n_recommendations=5):
  song_features = nX[song_index]

  #supervised model to predict genre
  predicted_genre = knn.predict(song_features.reshape(1, -1))[0]

  #Filter dataset for songs in that predicted genre
  genre_mask = data['track_genre'] == predicted_genre
  genre_data = data[genre_mask] #genre_data is data that only includes songs with the target genre
  genre_nX = nX[genre_mask] #genre_nX is nX that only includes songs with the target genre

  #Unsupervised model to rank similarity
  neigh = NearestNeighbors(n_neighbors=n_recommendations + 1)
  neigh.fit(genre_nX)

  #Find nearest neighbours
  distances, relative_indices = neigh.kneighbors(song_features.reshape(1,-1))
  #this finds the nearest neighbours in filtered genre_nX dataset
  #relative_indices here is the indices of closest to furthest neighbours (closest song = relative index of 0)

  #Map back to original dataframe index
  original_genre_indices = data[genre_mask].index.tolist() #makes the indices in a list
  temp_recommended_original_indices = [original_genre_indices[i] for i in relative_indices[0]] #maps relative index to actual index

  # Exclude the query song and take the top n_recommendations
  recommended_indices = [idx for idx in temp_recommended_original_indices if idx != song_index][:n_recommendations]

  return data.loc[recommended_indices, req_features]

print(classify_and_recommend(10, req_features=['album_name', 'track_name', 'duration_ms']))

               album_name                track_name  duration_ms
137  Eye To The Telescope            Suddenly I See       201706
488        Violent Femmes        Blister In The Sun       145706
71           America Town  Superman (It's Not Easy)       221693
128         Some Mad Hope        Come On Get Higher       215173
750           青春のエキサイトメント                 愛を伝えたいだとか       235240


Using the function `classify_and_recommend`, we then try to find songs with similar `album_name`, `loudness`, `acousticness`

In [11]:
print(classify_and_recommend(20, req_features=['album_name', 'loudness', 'acousticness']))

                                            album_name  loudness  acousticness
7                  We Sing. We Dance. We Steal Things.    -9.331        0.5590
6                          Is There Anybody Out There?    -8.822        0.8570
22   Is There Anybody Out There? - Track by Track C...    -8.822        0.8570
180                              I Need You More Today    -7.513        0.3600
371                                     Passenger Seat    -5.696        0.0456
